# Lab 10 — Landing.ai Agentic Document Extraction (ADE)

Questo notebook esplora l'SDK **Landing.ai ADE** nel contesto di GraphoLab.  
Usiamo documenti forensi (perizie grafologiche, testamenti, atti notarili) anziché utility bills.

## ADE in sintesi

| Operazione | Input | Output |
|---|---|---|
| **Parse** | PDF / immagine | Markdown strutturato + chunks con bounding box |
| **Extract** | Markdown + schema JSON | Campi chiave/valore + riferimenti visivi (grounding) |
| **Split** | Markdown | Segmenti classificati per tipo di documento |

## Fondamenta tecnologiche
- **Vision-First**: i documenti sono oggetti visivi; significato = layout + struttura + relazioni spaziali
- **Data-Centric**: modelli addestrati su dataset ampi e curati
- **Agentic**: il sistema pianifica, decide, agisce e verifica fino al raggiungimento di soglie di qualità

Il modello sottostante è **DPT** (Document Pre-trained Transformer): combina parsing testuale, rilevamento layout, ordine di lettura e ragionamento multimodale.

## Struttura
1. [Setup e autenticazione](#1)
2. [Parse — documento → markdown strutturato](#2)
3. [Esplora il ParseResponse (chunks, grounding, metadata)](#3)
4. [Extract — markdown → campi chiave/valore](#4)
5. [Esplora l'ExtractionMetadata (visual grounding)](#5)
6. [Wrapper GraphoLab: ade_pipeline()](#6)
7. [Schema personalizzato (JSON diretto)](#7)
8. [Schemi predefiniti GraphoLab](#8)
9. [Modelli disponibili: dpt-1 vs dpt-2](#9)

**Prerequisiti:**
```bash
pip install landingai-ade
```
Chiave in `.env` → `VISION_AGENT_API_KEY=...`

<a id="1"></a>

## 1. Setup e autenticazione

Import chiave dall'SDK Landing.ai:
- `LandingAIADE`: client per le chiamate API
- `ParseResponse`: tipo per i risultati di parsing
- `ExtractResponse`: tipo per i risultati di estrazione

In [1]:
import sys, os, json
from pathlib import Path
from IPython.display import display, HTML, Markdown

# Project root
ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))

# Load .env (VISION_AGENT_API_KEY, OPENAI_API_KEY, ...)
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

print("ROOT:", ROOT)
key = os.environ.get("VISION_AGENT_API_KEY", "")
print("VISION_AGENT_API_KEY configurata:", bool(key), f"({key[:8]}...)" if key else "")

ROOT: C:\Users\fanto\OneDrive\Documenti\AI\GraphoLab
VISION_AGENT_API_KEY configurata: True (ZTliY2Jw...)


In [2]:
# SDK Landing.ai ADE
from landingai_ade import LandingAIADE
from landingai_ade.types import ParseResponse, ExtractResponse
from landingai_ade.lib.schema_utils import pydantic_to_json_schema

print("landingai-ade importato correttamente")

landingai-ade importato correttamente


In [3]:
# Inizializza il client — legge VISION_AGENT_API_KEY automaticamente dall'ambiente
client = LandingAIADE()
print("Client ADE inizializzato")

Client ADE inizializzato


<a id="2"></a>

## 2. Parse — documento → markdown strutturato

La **Parse API** converte un documento in:
- **`chunks`**: regioni semantiche (testo, tabelle, figure, loghi, marginalia, attestazioni)
- **Bounding box**: coordinate di ciascun chunk sulla pagina
- **`markdown`**: rappresentazione testuale con ID chunk incorporati

Usando `dpt-2-latest` si ottiene la versione più aggiornata del modello DPT-2.

In [4]:
# --- Imposta qui il percorso del tuo documento forense ---
# Formati supportati: PDF, PNG, JPG, TIFF
SAMPLE_DOC = ROOT / "data" / "samples" / "testamento-luca-rossi.jpeg"

if not SAMPLE_DOC.exists():
    print(f"File non trovato: {SAMPLE_DOC}")
    print("Modifica la variabile SAMPLE_DOC con il percorso di un tuo documento.")
else:
    print(f"Documento: {SAMPLE_DOC.name} ({SAMPLE_DOC.stat().st_size / 1024:.1f} KB)")

Documento: testamento-luca-rossi.jpeg (113.9 KB)


In [5]:
print("Chiamata API Parse...")

parse_result: ParseResponse = client.parse(
    document=SAMPLE_DOC,
    model="dpt-2-latest"
)

print("Parse completato.")
print(f"  job_id:           {parse_result.metadata.job_id}")
print(f"  filename:         {parse_result.metadata.filename}")
print(f"  durata (ms):      {parse_result.metadata.duration_ms}")
print(f"  crediti usati:    {parse_result.metadata.credit_usage}")
print(f"  pagine totali:    {len(parse_result.splits)}")
print(f"  chunks totali:    {len(parse_result.chunks)}")
print(f"  markdown chars:   {len(parse_result.markdown)}")

Chiamata API Parse...
Parse completato.
  job_id:           a1a66de84e3947079b0c442938107538
  filename:         testamento-luca-rossi.jpeg
  durata (ms):      8896
  crediti usati:    3.0
  pagine totali:    1
  chunks totali:    7
  markdown chars:   1251


<a id="3"></a>

## 3. Esplora il ParseResponse

Il risultato del parse contiene la struttura completa del documento.  
Ogni chunk ha: `id`, `type`, `grounding.page`, `grounding.box`, `markdown`.

In [6]:
# Ispeziona i primi 5 chunk
parse_result.chunks[0:5]

[Chunk(id='4cafab91-79a4-498c-a595-0cd12411757b', grounding=ChunkGrounding(box=ParseGroundingBox(bottom=0.10493947565555573, left=0.15096931159496307, right=0.5895518064498901, top=0.06205997243523598), page=0), markdown="<a id='4cafab91-79a4-498c-a595-0cd12411757b'></a>\n\nTestamento Luca Rossi", type='text'),
 Chunk(id='0dec050a-1aa1-44b9-b5e7-478f4de2c538', grounding=ChunkGrounding(box=ParseGroundingBox(bottom=0.18645362555980682, left=0.15032702684402466, right=0.868891179561615, top=0.1241697296500206), page=0), markdown="<a id='0dec050a-1aa1-44b9-b5e7-478f4de2c538'></a>\n\nIo sottoscritto Luca Rossi, nato a Roma il 12 marzo 1950, residente in Corso Italia, 27, Napoli 41206, in piena facoltà mentale e di libera volontà, redigo questo mio testamento olografo.", type='text'),
 Chunk(id='dfeeea91-651a-42db-904c-0dd21c87996f', grounding=ChunkGrounding(box=ParseGroundingBox(bottom=0.24089500308036804, left=0.150048166513443, right=0.890597939491272, top=0.19429221749305725), page=0), m

In [7]:
# Dettagli del primo chunk
c = parse_result.chunks[0]
print(f"id:      {c.id}")
print(f"type:    {c.type}")
print(f"pagina:  {c.grounding.page}")
print(f"box:     {c.grounding.box}")

id:      4cafab91-79a4-498c-a595-0cd12411757b
type:    text
pagina:  0
box:     ParseGroundingBox(bottom=0.10493947565555573, left=0.15096931159496307, right=0.5895518064498901, top=0.06205997243523598)


In [8]:
# Quanti chunk per ogni tipo?
# Tipi possibili: text, table, figure, logo, marginalia, attestation
counts = {}
for chunk in parse_result.model_dump()["chunks"]:
    t = chunk["type"]
    counts[t] = counts.get(t, 0) + 1

print("Distribuzione chunk per tipo:")
for tipo, n in sorted(counts.items(), key=lambda x: -x[1]):
    bar = "█" * n
    print(f"  {tipo:<15} {n:3d}  {bar}")

Distribuzione chunk per tipo:
  text              6  ██████
  attestation       1  █


In [9]:
# Markdown top-level: contiene il documento intero con ID chunk incorporati
# Gli ID permettono il visual grounding (tracciare ogni valore estratto alla sua posizione)
print("MARKDOWN TOP-LEVEL (primi 1000 caratteri):")
print(parse_result.markdown[:1000])

MARKDOWN TOP-LEVEL (primi 1000 caratteri):
<a id='4cafab91-79a4-498c-a595-0cd12411757b'></a>

Testamento Luca Rossi

<a id='0dec050a-1aa1-44b9-b5e7-478f4de2c538'></a>

Io sottoscritto Luca Rossi, nato a Roma il 12 marzo 1950, residente in Corso Italia, 27, Napoli 41206, in piena facoltà mentale e di libera volontà, redigo questo mio testamento olografo.

<a id='dfeeea91-651a-42db-904c-0dd21c87996f'></a>

Nomino mio erede universale Giovanni Costa, residente in Piazza Duomo, 78, Napoli 39030,
al quale lascio tutti i miei beni mobili e immobili, conti correnti e investimenti.

<a id='36765869-c3a6-4f4d-90ca-34defdf3f3de'></a>

In particolare, gli lascio il possesso del mio conto bancario con IBAN
IT24A16351628369207418068 e la proprietà dell'appartamento sito in Via Garibaldi, 21,
Milano 20436.

<a id='1ee24fc1-bf29-484e-b9d6-46a8b54a3a34'></a>

Revoco ogni mio precedente testamento.

<a id='6a33a05e-a78a-4a64-a981-a7f34a5d4769'></a>

Fatto, scritto e firmato di mio pugno in Firenze, il 

In [10]:
# Markdown a livello di chunk — le tabelle sono rese come HTML con ID per cella
# Cerca il primo chunk di tipo 'table'
table_chunks = [c for c in parse_result.chunks if c.type == "table"]

if table_chunks:
    print(f"Trovate {len(table_chunks)} tabelle. Prima tabella (raw markdown):")
    print(table_chunks[0].markdown)
    print("\nResa come HTML:")
    display(HTML(table_chunks[0].markdown))
else:
    print("Nessuna tabella trovata in questo documento.")
    print("Markdown del primo chunk text:")
    text_chunks = [c for c in parse_result.chunks if c.type == "text"]
    if text_chunks:
        print(text_chunks[0].markdown)

Nessuna tabella trovata in questo documento.
Markdown del primo chunk text:
<a id='4cafab91-79a4-498c-a595-0cd12411757b'></a>

Testamento Luca Rossi


<a id="4"></a>

## 4. Extract — markdown → campi chiave/valore

La **Extract API** usa il markdown del passo precedente e uno schema JSON per estrarre campi specifici.

Lo schema supporta:
- Oggetti annidati (es. `account_summary` con sottocampi)
- Tipi multipli: `string`, `number`, `boolean`, `array`
- Descrizioni ricche — più la descrizione è precisa, migliore è l'estrazione

> **Nota**: lo schema deve essere passato come **stringa JSON**, non come dict Python.
> Usa `pydantic_to_json_schema(MyModel)` per i modelli Pydantic, o `json.dumps(schema_dict)` per dizionari.

In [11]:
# Definisci lo schema come dizionario Python, poi converti in JSON string
schema_dict = {
    "type": "object",
    "title": "Perizia Forense — Campi Principali",
    "properties": {
        "intestazione": {
            "type": "object",
            "title": "Dati intestazione",
            "properties": {
                "numero_caso": {
                    "type": "string",
                    "description": "Numero o codice univoco che identifica il caso (fascicolo, repertorio)."
                },
                "perito_nome": {
                    "type": "string",
                    "description": "Nome completo del perito grafologico che ha redatto la perizia."
                },
                "perito_qualifica": {
                    "type": "string",
                    "description": "Titolo professionale e qualifiche del perito (es. CTU, iscrizione albo)."
                },
                "data_firma": {
                    "type": "string",
                    "description": "Data in cui la perizia è stata firmata dal perito. Non confondere con data incarico o ricezione materiale."
                }
            }
        },
        "materiale": {
            "type": "object",
            "title": "Materiale esaminato",
            "properties": {
                "data_ricezione": {
                    "type": "string",
                    "description": "Data di ricezione fisica del materiale da parte del perito."
                },
                "trasmittente": {
                    "type": "string",
                    "description": "Nome e qualifica di chi ha trasmesso il materiale (avvocato, notaio, tribunale)."
                },
                "num_documenti": {
                    "type": "number",
                    "description": "Numero totale di documenti/reperti nell'elenco del materiale esaminato."
                },
                "presente_originale": {
                    "type": "boolean",
                    "description": "Il perito ha esaminato l'originale del documento (true) o solo una copia/scansione (false)?"
                }
            }
        },
        "conclusioni": {
            "type": "object",
            "title": "Conclusioni peritali",
            "properties": {
                "livello_certezza": {
                    "type": "string",
                    "description": "Livello di certezza espresso nelle conclusioni (es. 'con alta probabilità tecnica', 'con certezza', scala ENFSI)."
                },
                "esito": {
                    "type": "string",
                    "description": "Esito conclusivo della perizia: autentico, falso, non determinabile, o simile."
                },
                "scala_enfsi_usata": {
                    "type": "boolean",
                    "description": "La relazione usa esplicitamente la scala ENFSI (da 'certamente' a 'probabilmente non')?"
                }
            }
        }
    }
}

# Converti in stringa JSON — formato richiesto dall'API
schema_json = json.dumps(schema_dict)
print("Schema definito con", len(schema_dict["properties"]), "sezioni principali")

Schema definito con 3 sezioni principali


In [12]:
print("Chiamata API Extract...")

# NOTA: l'input è parse_result.markdown (markdown top-level dal passo Parse)
# model="extract-latest" usa sempre la versione più aggiornata
extraction_result: ExtractResponse = client.extract(
    schema=schema_json,
    markdown=parse_result.markdown,
    model="extract-latest"
)

print("Estrazione completata.")
print(f"  job_id:        {extraction_result.metadata.job_id}")
print(f"  durata (ms):   {extraction_result.metadata.duration_ms}")
print(f"  crediti usati: {extraction_result.metadata.credit_usage}")

Chiamata API Extract...
Estrazione completata.
  job_id:        5fc9f7f2851745b18ffc5e07bc088190
  durata (ms):   77117
  crediti usati: 0.5522


In [13]:
# Visualizza i valori estratti
print("=== CAMPI ESTRATTI ===")
print(json.dumps(
    extraction_result.extraction if hasattr(extraction_result, 'extraction') 
    else extraction_result.model_dump().get('extraction', {}),
    ensure_ascii=False, indent=2
))

=== CAMPI ESTRATTI ===
{
  "intestazione": {
    "numero_caso": null,
    "perito_nome": null,
    "perito_qualifica": null,
    "data_firma": null
  },
  "materiale": {
    "data_ricezione": null,
    "trasmittente": null,
    "num_documenti": 1,
    "presente_originale": false
  },
  "conclusioni": {
    "livello_certezza": null,
    "esito": null,
    "scala_enfsi_usata": false
  }
}


<a id="5"></a>

## 5. ExtractionMetadata — Visual Grounding

Ogni campo estratto ha dei **riferimenti** ai chunk/celle sorgente nel documento originale.  
Questo è il **visual grounding**: traccia ogni valore alla sua posizione esatta nella pagina.

- ID brevi tipo `0-a` → celle di tabella
- UUID lunghi → chunk di testo o figure

Questo è fondamentale per le perizie forensi: ogni valore estratto è *citabile e verificabile*.

In [14]:
# Visualizza i metadati di grounding
print("=== EXTRACTION METADATA (riferimenti visivi) ===")
raw = extraction_result.model_dump()
print(json.dumps(raw.get('extraction_metadata', {}), ensure_ascii=False, indent=2))

=== EXTRACTION METADATA (riferimenti visivi) ===
{
  "intestazione": {
    "numero_caso": {
      "references": [],
      "value": null
    },
    "perito_nome": {
      "references": [],
      "value": null
    },
    "perito_qualifica": {
      "references": [],
      "value": null
    },
    "data_firma": {
      "references": [],
      "value": null
    }
  },
  "materiale": {
    "data_ricezione": {
      "references": [],
      "value": null
    },
    "trasmittente": {
      "references": [],
      "value": null
    },
    "num_documenti": {
      "references": [
        "4cafab91-79a4-498c-a595-0cd12411757b"
      ],
      "value": 1
    },
    "presente_originale": {
      "references": [],
      "value": false
    }
  },
  "conclusioni": {
    "livello_certezza": {
      "references": [],
      "value": null
    },
    "esito": {
      "references": [],
      "value": null
    },
    "scala_enfsi_usata": {
      "references": [],
      "value": false
    }
  }
}


In [15]:
# Analisi: quanti campi hanno almeno un riferimento visivo?
meta = raw.get('extraction_metadata', {})
extraction = raw.get('extraction', {})

print("Campo                          | Valore estratto              | Riferimenti")
print("-" * 80)

def _flatten(d, prefix=""):
    """Flatten nested dict for display."""
    for k, v in d.items():
        full_key = f"{prefix}.{k}" if prefix else k
        if isinstance(v, dict) and not any(kk in v for kk in ("value", "references")):
            yield from _flatten(v, full_key)
        else:
            yield full_key, v

for field_path, val in _flatten(extraction):
    # Look up metadata for this field
    parts = field_path.split(".")
    m = meta
    for p in parts:
        m = m.get(p, {}) if isinstance(m, dict) else {}
    refs = m.get("references", []) if isinstance(m, dict) else []
    val_str = str(val)[:30] if val is not None else "—"
    ref_str = str(len(refs)) + " ref" if refs else "nessuno"
    print(f"{field_path:<35} | {val_str:<28} | {ref_str}")

Campo                          | Valore estratto              | Riferimenti
--------------------------------------------------------------------------------
intestazione.numero_caso            | —                            | nessuno
intestazione.perito_nome            | —                            | nessuno
intestazione.perito_qualifica       | —                            | nessuno
intestazione.data_firma             | —                            | nessuno
materiale.data_ricezione            | —                            | nessuno
materiale.trasmittente              | —                            | nessuno
materiale.num_documenti             | 1                            | 1 ref
materiale.presente_originale        | False                        | nessuno
conclusioni.livello_certezza        | —                            | nessuno
conclusioni.esito                   | —                            | nessuno
conclusioni.scala_enfsi_usata       | False                        | nessun

<a id="6"></a>

## 6. Wrapper GraphoLab: `ade_pipeline()`

Il modulo `core/docextract.py` espone wrapper di alto livello che incapsulano parse + extract in una sola chiamata, con schemi Pydantic predefiniti per il dominio forense.

In [16]:
from core.docextract import ade_pipeline, ForensicReportSchema

result = ade_pipeline(SAMPLE_DOC, ForensicReportSchema)

if result.ok:
    print(f"Pipeline OK — markdown: {len(result.markdown)} chars, chunks: {len(result.chunks)}")
    filled = {k: v for k, v in result.fields.items() if v is not None}
    print(f"Campi ENFSI estratti: {len(filled)}/20")
    print()
    print(json.dumps(filled, ensure_ascii=False, indent=2))
else:
    print(f"Errore: {result.error}")

Pipeline OK — markdown: 1251 chars, chunks: 7
Campi ENFSI estratti: 2/20

{
  "caso_identificatore": "Testamento Luca Rossi",
  "data_firma_relazione": "15 giugno 2024"
}


In [17]:
# Esplora i metadati di grounding via AdeResult
if result.ok and result.extraction_metadata:
    print("=== EXTRACTION METADATA via AdeResult ===")
    print(json.dumps(result.extraction_metadata, ensure_ascii=False, indent=2))

=== EXTRACTION METADATA via AdeResult ===
{
  "caso_identificatore": {
    "references": [
      "7c1cc176-16b2-4184-89e2-239c89a8857a"
    ],
    "value": "Testamento Luca Rossi"
  },
  "laboratorio_nome": {
    "references": [],
    "value": null
  },
  "laboratorio_indirizzo": {
    "references": [],
    "value": null
  },
  "esaminatore_nome": {
    "references": [],
    "value": null
  },
  "esaminatore_qualifiche": {
    "references": [],
    "value": null
  },
  "firma_esaminatore": {
    "references": [],
    "value": null
  },
  "data_firma_relazione": {
    "references": [
      "0bd39ac7-76c9-422c-a209-b96dc226a12c"
    ],
    "value": "15 giugno 2024"
  },
  "data_ricezione_materiale": {
    "references": [],
    "value": null
  },
  "trasmittente_nome": {
    "references": [],
    "value": null
  },
  "trasmittente_qualifica": {
    "references": [],
    "value": null
  },
  "elenco_materiale": {
    "references": [],
    "value": null
  },
  "stato_materiale_ricezione": {

<a id="7"></a>

## 7. Schema personalizzato (JSON diretto)

Puoi definire qualsiasi schema per adattarsi al tuo specifico tipo di documento.  
Qui estraiamo campi da un testamento olografo usando un dizionario schema.

In [18]:
from pydantic import BaseModel, Field
from typing import Optional
from core.docextract import ade_extract

# Approccio 1: schema Pydantic → automaticamente convertito in JSON string
class TestamentoSchema(BaseModel):
    testatore: Optional[str] = Field(None, description="Nome completo del testatore")
    data_testamento: Optional[str] = Field(None, description="Data di redazione del testamento")
    eredi: Optional[list[str]] = Field(None, description="Lista dei nomi degli eredi nominati")
    lasciti_speciali: Optional[str] = Field(None, description="Lasciti particolari a persone specifiche diversi dalla quota ereditaria")
    testimoni: Optional[list[str]] = Field(None, description="Nomi dei testimoni presenti alla firma")
    notaio: Optional[str] = Field(None, description="Nome del notaio rogante se presente")
    luogo: Optional[str] = Field(None, description="Luogo di redazione del testamento")

sample_testamento = """
Io, Carlo Verdi, nato a Torino il 12/05/1940, in piena capacità mentale,
dispongo delle mie ultime volontà:

Nomino eredi universali i miei figli Maria Verdi e Luca Verdi in parti uguali.
Lascio l'appartamento di Via Manzoni 5, Torino, a mia nipote Sofia Esposito.
Lascio la collezione di orologi al mio amico Dott. Roberto Ferrari.

Testimoni: Sig. Antonio Ferraro e Sig.ra Rosa Conti.
Notaio rogante: Dott. Paolo Greco, Notaio in Torino, Via Dante 8.
Torino, 5 gennaio 2024.
"""

fields, extraction_metadata = ade_extract(sample_testamento, TestamentoSchema)
print("=== Campi estratti ===")
print(json.dumps(fields, ensure_ascii=False, indent=2))

=== Campi estratti ===
{
  "testatore": "Carlo Verdi",
  "data_testamento": "5 gennaio 2024",
  "eredi": [
    "Maria Verdi",
    "Luca Verdi"
  ],
  "lasciti_speciali": "L'appartamento di Via Manzoni 5, Torino, a mia nipote Sofia Esposito. La collezione di orologi al mio amico Dott. Roberto Ferrari.",
  "testimoni": [
    "Sig. Antonio Ferraro",
    "Sig.ra Rosa Conti"
  ],
  "notaio": "Dott. Paolo Greco",
  "luogo": "Torino"
}


In [19]:
# Approccio 2: schema JSON diretto (dizionario → json.dumps)
schema_dict_custom = {
    "type": "object",
    "title": "Atto Notarile — Campi Essenziali",
    "properties": {
        "numero_repertorio": {
            "type": "string",
            "description": "Numero di repertorio notarile dell'atto"
        },
        "data_atto": {
            "type": "string",
            "description": "Data di stipula dell'atto notarile"
        },
        "parti": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Nomi di tutte le parti contraenti dell'atto"
        },
        "valore_immobile": {
            "type": "number",
            "description": "Valore economico dell'immobile in euro se presente nell'atto"
        }
    }
}

fields2, _ = ade_extract(sample_testamento, json.dumps(schema_dict_custom))
print("=== Schema JSON diretto ===")
print(json.dumps(fields2, ensure_ascii=False, indent=2))

=== Schema JSON diretto ===
{
  "numero_repertorio": null,
  "data_atto": "5 gennaio 2024",
  "parti": [
    "Carlo Verdi",
    "Maria Verdi",
    "Luca Verdi",
    "Sofia Esposito",
    "Dott. Roberto Ferrari"
  ],
  "valore_immobile": null
}


<a id="8"></a>

## 8. Schemi predefiniti GraphoLab

Il modulo `core/docextract.py` include tre schemi Pydantic pronti all'uso per il dominio forense.

In [20]:
from core.docextract import SCHEMA_REGISTRY

for name, schema_cls in SCHEMA_REGISTRY.items():
    fields_list = list(schema_cls.model_fields.keys())
    print(f"\n{'='*60}")
    print(f"Schema: {name}")
    print(f"Classe: {schema_cls.__name__} — {len(fields_list)} campi")
    print(f"{'='*60}")
    for f in fields_list:
        info = schema_cls.model_fields[f]
        desc = info.description or ""
        print(f"  {f:<35} {desc[:55]}")


Schema: Perizia Forense (ENFSI)
Classe: ForensicReportSchema — 20 campi
  caso_identificatore                 Identificatore univoco del caso (numero fascicolo, repe
  laboratorio_nome                    Nome del laboratorio o professionista
  laboratorio_indirizzo               Indirizzo o recapito del laboratorio
  esaminatore_nome                    Nome del perito esaminatore
  esaminatore_qualifiche              Qualifiche, titoli, iscrizioni albo del perito
  firma_esaminatore                   Presenza della firma del perito (sì/no/autografa/digita
  data_firma_relazione                Data di firma della relazione peritale
  data_ricezione_materiale            Data di ricezione del materiale esaminato
  trasmittente_nome                   Nome di chi ha trasmesso il materiale
  trasmittente_qualifica              Qualifica del trasmittente (avvocato, notaio, tribunale
  elenco_materiale                    Elenco dei documenti/reperti esaminati
  stato_materiale_ricezione      

In [21]:
# Visualizza lo schema JSON generato da pydantic_to_json_schema
# È ciò che viene passato all'API Extract
from core.docextract import ForensicReportSchema

schema_str = pydantic_to_json_schema(ForensicReportSchema)
schema_parsed = json.loads(schema_str)
print(f"Schema ForensicReportSchema ({len(schema_parsed['properties'])} campi):")
print(json.dumps(schema_parsed, ensure_ascii=False, indent=2))

Schema ForensicReportSchema (20 campi):
{
  "description": "ENFSI BPM-FHX-01 Ed.03 — 20 required elements for a forensic report.",
  "properties": {
    "caso_identificatore": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Identificatore univoco del caso (numero fascicolo, repertorio)",
      "title": "Caso Identificatore"
    },
    "laboratorio_nome": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Nome del laboratorio o professionista",
      "title": "Laboratorio Nome"
    },
    "laboratorio_indirizzo": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Indirizzo o recapito del laboratorio",
      "title": "Laboratorio Indirizzo"
    }

<a id="9"></a>

## 9. Modelli disponibili: `dpt-1` vs `dpt-2`

| Modello | Quando usarlo |
|---|---|
| `dpt-2-latest` | Default — ottimo per testo, tabelle, form, firme, timbri |
| `dpt-1-latest` | Illustrazioni e infografiche senza testo (descrive le immagini) |
| `extract-latest` | Extract API — sempre la versione più aggiornata |

Per documenti forensi con firme e timbri, `dpt-2-latest` è la scelta corretta:  
riconosce le `attestation` chunk (firme, timbri con testo curvo) che `dpt-1` non gestisce.

In [22]:
# Esempio: parsing con dpt-1-latest per confronto
# Utile per documenti che contengono prevalentemente illustrazioni (es. esame calligrafico visivo)

if SAMPLE_DOC.exists():
    from core.docextract import ade_parse
    
    print("Parsing con dpt-1-latest...")
    md_dpt1, chunks_dpt1 = ade_parse(SAMPLE_DOC, model="dpt-1-latest")
    
    print("Parsing con dpt-2-latest...")
    md_dpt2, chunks_dpt2 = ade_parse(SAMPLE_DOC, model="dpt-2-latest")
    
    print(f"\ndpt-1: {len(md_dpt1)} chars, {len(chunks_dpt1)} chunks")
    print(f"dpt-2: {len(md_dpt2)} chars, {len(chunks_dpt2)} chunks")
    
    # Distribuzione chunk per tipo — dpt-1 vs dpt-2
    def chunk_counts(chunks):
        counts = {}
        for c in chunks:
            t = c.get('type', 'unknown') if isinstance(c, dict) else c['type']
            counts[t] = counts.get(t, 0) + 1
        return counts
    
    print("\nDistribuzione chunk:")
    print(f"{'Tipo':<15} {'dpt-1':>6} {'dpt-2':>6}")
    all_types = set(list(chunk_counts(chunks_dpt1).keys()) + list(chunk_counts(chunks_dpt2).keys()))
    for t in sorted(all_types):
        n1 = chunk_counts(chunks_dpt1).get(t, 0)
        n2 = chunk_counts(chunks_dpt2).get(t, 0)
        print(f"{t:<15} {n1:>6} {n2:>6}")
else:
    print("Imposta SAMPLE_DOC con un documento valido per eseguire questo confronto.")

Parsing con dpt-1-latest...
Parsing con dpt-2-latest...

dpt-1: 1760 chars, 7 chunks
dpt-2: 1251 chars, 7 chunks

Distribuzione chunk:
Tipo             dpt-1  dpt-2
attestation          0      1
figure               1      0
text                 6      6


## Riepilogo

| Concetto | Descrizione |
|---|---|
| **Parse API** | Converte documenti in markdown strutturato con chunks e bounding box |
| **Extract API** | Estrae coppie chiave/valore via schema JSON, con riferimenti per visual grounding |
| **Modelli DPT** | `dpt-2-latest` per testo/tabelle/firme; `dpt-1-latest` per illustrazioni |
| **Tipi di chunk** | `text`, `table`, `figure`, `logo`, `marginalia`, `attestation` (firme e timbri) |
| **Visual Grounding** | Ogni valore estratto referenzia il chunk sorgente nel documento |
| **GraphoLab** | `core/docextract.py` espone `ade_parse`, `ade_extract`, `ade_pipeline` + 3 schemi forensi |

### Integrazione nel progetto
- **Tab Gradio**: `📄 Estrazione Strutturata` — carica un documento, scegli lo schema, ottieni i campi
- **Compliance checker**: usa ADE per pre-estrarre i 20 campi ENFSI prima della verifica LLM
- **Tracciabilità forense**: i `references` di `extraction_metadata` consentono di citare la fonte esatta di ogni valore estratto